# 02 · State

**What this teaches:** the four ways to declare the calculator's state, and how
reducers control the way updates are applied.

## What is state?

From the [Graph API docs](https://docs.langchain.com/oss/python/langgraph/graph-api#state):

> **State** is "a shared data structure that represents the current snapshot of your
> application."

The [schema](https://docs.langchain.com/oss/python/langgraph/graph-api#schema) you pass
to `StateGraph(...)` "serves as the input schema for all Nodes and Edges in the graph".

The docs support three schema types in Python — **`TypedDict`**, **Pydantic
`BaseModel`**, and **dataclass**.

In [1]:
import operator
from dataclasses import dataclass, field
from typing import Annotated

from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, ValidationError, field_validator
from typing_extensions import TypedDict

In [2]:
def build(schema, node):
    """Compile the same one-node calculator for whichever schema we are demonstrating."""
    builder = StateGraph(schema)
    builder.add_node("divide", node)
    builder.add_edge(START, "divide")
    builder.add_edge("divide", END)
    return builder.compile()

## 1. TypedDict

Type hints on the keys and nothing more. There is **no runtime checking** — the hints
are for your editor, and a wrong value at runtime passes straight through.

In [3]:
class DictState(TypedDict):
    a: float
    b: float
    result: float


def dict_divide(state: DictState) -> dict:
    return {"result": state["a"] / state["b"]}


build(DictState, dict_divide).invoke({"a": 9, "b": 2})

{'a': 9, 'b': 2, 'result': 4.5}

In [4]:
# a divisor of 0 is not the schema's problem — TypedDict is hints only
try:
    build(DictState, dict_divide).invoke({"a": 9, "b": 0})
except ZeroDivisionError as e:
    print("ZeroDivisionError:", e)

ZeroDivisionError: division by zero


## 2. Pydantic BaseModel — validated

A Pydantic model validates the **input** to the graph at runtime, so a bad divisor is
rejected before any node runs.

Nodes are handed the model instance but still return a plain `dict` of updates.

In [5]:
class ModelState(BaseModel):
    a: float
    b: float
    result: float = 0.0

    @field_validator("b")
    @classmethod
    def not_zero(cls, v: float) -> float:
        if v == 0:
            raise ValueError("cannot divide by zero")
        return v


def model_divide(state: ModelState) -> dict:
    print(f"got a {type(state).__name__}: attribute access -> {state.a} / {state.b}")
    return {"result": state.a / state.b}


build(ModelState, model_divide).invoke({"a": 9, "b": 2})

got a ModelState: attribute access -> 9.0 / 2.0


{'a': 9, 'b': 2, 'result': 4.5}

Now the divisor that `TypedDict` waved through:

In [6]:
try:
    build(ModelState, model_divide).invoke({"a": 9, "b": 0})
except ValidationError as e:
    print("ValidationError:", e.errors()[0]["msg"])

ValidationError: Value error, cannot divide by zero


In [7]:
try:
    build(ModelState, model_divide).invoke({"a": 9, "b": "two"})
except ValidationError as e:
    print("ValidationError:", e.errors()[0]["msg"])

ValidationError: Input should be a valid number, unable to parse string as a number


## 3. Dataclass

Attribute access like Pydantic, no validation like `TypedDict`. Mutable defaults need
`field(default_factory=...)`.

In [8]:
@dataclass
class DataState:
    a: float = 0.0
    b: float = 1.0
    result: float = 0.0
    steps: list[str] = field(default_factory=list)


def data_divide(state: DataState) -> dict:
    print(f"got a {type(state).__name__}: attribute access -> {state.a} / {state.b}")
    return {"result": state.a / state.b}


build(DataState, data_divide).invoke(DataState(a=9, b=2))

got a DataState: attribute access -> 9 / 2


{'a': 9, 'b': 2, 'result': 4.5, 'steps': []}

### Choosing one

| | Access | Runtime validation | Defaults | Use when |
| --- | --- | --- | --- | --- |
| `TypedDict` | `state["k"]` | none | no | default choice, lightest |
| Pydantic `BaseModel` | `state.k` | yes | yes | input comes from outside your code |
| `@dataclass` | `state.k` | none | yes | attributes without a Pydantic dependency |

One schema type per `StateGraph`, not per node.

## 4. Reducers

Every example so far *replaced* `result`. That is the default:

> [**Reducers**](https://docs.langchain.com/oss/python/langgraph/graph-api#reducers) are
> "reducer functions which specify how to apply updates to the state."

Each key gets its own reducer. With none, the update overwrites. Attach one with
`Annotated[<type>, <reducer fn>]` and the update is merged instead — which is how the
calculator keeps a tape of what it did.

In [9]:
class Overwrite(TypedDict):
    steps: list[str]                              # no reducer -> replace


class Accumulate(TypedDict):
    steps: Annotated[list[str], operator.add]     # reducer -> concatenate


def record(state) -> dict:
    return {"steps": ["x2"]}


print("overwrite: ", build(Overwrite, record).invoke({"steps": ["+10"]}))
print("accumulate:", build(Accumulate, record).invoke({"steps": ["+10"]}))

overwrite:  {'steps': ['x2']}
accumulate: {'steps': ['+10', 'x2']}


Same node, same input, different annotation — the whole difference between a tape that
remembers and one that forgets.

Reducers are not only for lists. Any function `(current, update) -> new` works:

In [10]:
def keep_max(current: float, update: float) -> float:
    return max(current, update)


class Calculator(TypedDict):
    steps: Annotated[list[str], operator.add]
    largest_result: Annotated[float, keep_max]


def divide_and_record(state: Calculator) -> dict:
    return {"steps": ["9 / 2"], "largest_result": 4.5}


print(build(Calculator, divide_and_record).invoke({"steps": [], "largest_result": 100.0}))

{'steps': ['9 / 2'], 'largest_result': 100.0}


`largest_result` stayed at 100 because `keep_max` chose it over the node's 4.5. The
node did not have to know that rule — the schema enforced it.

Reducers matter most when **two nodes write the same key at once**, which is what
parallel branches do in [03-edges.ipynb](./03-edges.ipynb).

## Partial updates, once more

A node returns only the keys it touches, which is what keeps nodes independent.

In [11]:
class Mixed(TypedDict):
    steps: Annotated[list[str], operator.add]
    result: float
    label: str


def square(state: Mixed) -> dict:
    return {"result": state["result"] ** 2, "steps": ["squared"]}


build(Mixed, square).invoke({"steps": ["seed"], "result": 7, "label": "kept"})

{'steps': ['seed', 'squared'], 'result': 49, 'label': 'kept'}

## The same idea in TypeScript

LangGraph's JavaScript library uses [Zod](https://zod.dev) where Python uses
`TypedDict`, with reducers attached per field.

> **TypeScript, not runnable here** — this notebook has a Python kernel.

From the [JS Graph API docs](https://docs.langchain.com/oss/javascript/langgraph/graph-api):

```typescript
import { StateSchema, ReducedValue } from "@langchain/langgraph";
import { z } from "zod/v4";

const CalculatorState = new StateSchema({
  result: z.number().default(0),

  // the equivalent of Annotated[list[str], operator.add]
  steps: new ReducedValue(
    z.array(z.string()).default(() => []),
    {
      inputSchema: z.string(),
      reducer: (current, update) => [...current, update],
    }
  ),
});
```

| Python | TypeScript |
| --- | --- |
| `class State(TypedDict)` | `new StateSchema({ ... })` |
| `steps: list[str]` | `steps: z.array(z.string())` |
| `Annotated[list[str], operator.add]` | `new ReducedValue(schema, { reducer })` |
| `field(default_factory=list)` | `.default(() => [])` |
| Pydantic validation | Zod validation (built in) |

# Exercise